# Tutorial 14 — GRPO: Group Relative Policy Optimization

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part IV — Alignment**  
**Follows:** Tutorial 13 (Reward Model Training)  
**Precedes:** Tutorial 15 (Quantization)

---

## What This Tutorial Covers

GRPO (Shao et al., 2024) is the algorithm behind DeepSeek-R1 and a growing
number of reasoning-capable language models. It is a policy gradient method
— it improves the policy by sampling responses, scoring them, and nudging
the policy toward higher-scoring outputs. But unlike PPO, it requires no
value network and no per-token advantage estimation. The advantage of a
response is computed entirely from the rewards of its siblings — other
responses to the same prompt generated in the same batch.

This tutorial builds GRPO from first principles. No RL background is assumed.
Every concept is introduced before it is used.

Topics:

1. **Policy gradients from scratch** — the REINFORCE estimator, why it is
   an unbiased gradient estimate, why it has high variance.
2. **Baselines and advantages** — how subtracting a baseline reduces
   variance without introducing bias. The group baseline.
3. **The GRPO objective** — the full loss, the clipping term, the KL
   penalty. Comparison to PPO.
4. **Implementation** — sampling, scoring, the forward pass, the loss.
5. **The training loop** — the GRPO loop structure, which differs
   fundamentally from SFT.
6. **Monitoring GRPO** — reward, KL divergence, response entropy,
   the collapse diagnostic.
7. **Reward hacking** — what it looks like, how to detect it early,
   how to prevent it.

---

## 1. Policy Gradients From Scratch

A language model is a **policy**: given a prompt $x$, it produces a
probability distribution over responses $y$. We want to find the policy
parameters $\theta$ that maximize expected reward:

$$J(\theta) = \mathbb{E}_{y \sim \pi_\theta(\cdot \mid x)}\left[r(x, y)\right]$$

To maximize $J(\theta)$ with gradient ascent, we need $\nabla_\theta J(\theta)$.
The problem: the expectation is over $\pi_\theta$ itself — as $\theta$ changes,
the distribution we are integrating over changes. We cannot just move the
gradient inside the expectation.

The **log-derivative trick**[^logderiv] solves this:

[^logderiv]: The trick uses the identity $\nabla_\theta \pi_\theta = \pi_\theta \nabla_\theta \log \pi_\theta$, which lets us move the gradient inside the expectation even though the expectation is over $\pi_\theta$ itself. This converts an intractable distributional derivative into an expectation of a tractable gradient — computable via Monte Carlo sampling.

$$\nabla_\theta \mathbb{E}_{y \sim \pi_\theta}[r(y)]
= \mathbb{E}_{y \sim \pi_\theta}\left[r(y) \cdot \nabla_\theta \log \pi_\theta(y \mid x)\right]$$

**Derivation:**

$$\nabla_\theta \mathbb{E}_{y \sim \pi_\theta}[r(y)]
= \nabla_\theta \sum_y r(y) \pi_\theta(y \mid x)
= \sum_y r(y) \nabla_\theta \pi_\theta(y \mid x)$$

Use the identity $\nabla_\theta \pi_\theta = \pi_\theta \nabla_\theta \log \pi_\theta$:

$$= \sum_y r(y) \pi_\theta(y \mid x) \nabla_\theta \log \pi_\theta(y \mid x)
= \mathbb{E}_{y \sim \pi_\theta}\left[r(y) \cdot \nabla_\theta \log \pi_\theta(y \mid x)\right]$$

This is the **REINFORCE estimator** (Williams, 1992). We can approximate
it by sampling $G$ responses and averaging:

$$\nabla_\theta J(\theta) \approx \frac{1}{G} \sum_{i=1}^{G} r(y_i) \cdot \nabla_\theta \log \pi_\theta(y_i \mid x)$$

In PyTorch, $\nabla_\theta \log \pi_\theta(y \mid x)$ is the gradient of
the log-probability of the sampled response with respect to the parameters.
The reward $r(y_i)$ is just a scalar multiplier on that gradient.

**Intuition:** if response $y_i$ received a high reward, we increase the
probability of generating $y_i$ (positive gradient). If it received a low
reward, we decrease it. The magnitude of the update is proportional to the
reward.

### The variance problem

REINFORCE has high variance because the reward $r(y)$ can vary enormously
across samples. A response might score 10.0 in one batch and 0.1 in the
next. These large fluctuations mean the gradient estimate is noisy —
training is slow and unstable.

The fix is a **baseline**.

---

## 2. Baselines and Advantages

If we subtract any function $b(x)$ that does not depend on $y$ from the
reward, the gradient estimate remains unbiased:

$$\nabla_\theta J(\theta) = \mathbb{E}_{y \sim \pi_\theta}\left[(r(y) - b(x)) \cdot \nabla_\theta \log \pi_\theta(y \mid x)\right]$$

**Proof that the bias term vanishes:**

$$\mathbb{E}_{y \sim \pi_\theta}\left[b(x) \cdot \nabla_\theta \log \pi_\theta(y \mid x)\right]
= b(x) \mathbb{E}_{y \sim \pi_\theta}\left[\nabla_\theta \log \pi_\theta(y \mid x)\right]
= b(x) \cdot \nabla_\theta \underbrace{\sum_y \pi_\theta(y \mid x)}_{=1} = 0$$

The quantity $A(x, y) = r(y) - b(x)$ is called the **advantage**: how much
better is this particular response compared to the baseline expectation?

Choosing $b(x) = \mathbb{E}_{y \sim \pi_\theta}[r(y)]$ (the expected reward
for this prompt) minimizes variance. GRPO approximates this expectation
using the $G$ sampled responses:

$$b(x) = \frac{1}{G} \sum_{i=1}^{G} r(y_i)$$

$$A_i = r(y_i) - \frac{1}{G} \sum_{j=1}^{G} r(y_j)$$

This is the **group relative advantage**: response $i$'s advantage is how
much better it is than the average of all $G$ responses to the same prompt.
[No value network needed. No bootstrapping. Just the mean of the group.]{.mark}

GRPO also normalizes the advantages by the group standard deviation:

$$\hat{A}_i = \frac{r(y_i) - \text{mean}(r)}{\text{std}(r) + \epsilon}$$

This keeps advantage magnitudes consistent across prompts with different
reward scales.

In [ ]:
import torch
import numpy as np

def compute_group_advantages(rewards: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Compute normalized group relative advantages.

    Args:
        rewards: (G,) reward scores for G responses to the same prompt
    Returns:
        advantages: (G,) normalized advantages
    """
    mean = rewards.mean()
    std  = rewards.std()
    return (rewards - mean) / (std + eps)

---

## 3. The GRPO Objective

### The clipped surrogate

REINFORCE directly updates the policy in the direction of high-advantage
responses. The problem: a single large gradient step can move the policy
too far — destroying the coherent language model behavior that makes
responses legible in the first place.

PPO solves this with a **clipped surrogate objective** that limits how
much the policy can change in a single update. GRPO uses the same mechanism.

Let [$\rho_i(\theta)$ be the **importance ratio**:]{.underline} how much more (or less)
likely is response $y_i$ under the new policy $\pi_\theta$ compared to
the old policy $\pi_{\theta_\text{old}}$ that generated it:

$$\rho_i(\theta) = \frac{\pi_\theta(y_i \mid x)}{\pi_{\theta_\text{old}}(y_i \mid x)}$$

If $\rho_i > 1$: the new policy assigns higher probability to $y_i$ than
when it was sampled. If $\rho_i < 1$: lower probability.

The clipped surrogate objective for a single response:

$$\mathcal{L}_{\text{clip},i}(\theta) = \min\!\left(\rho_i \hat{A}_i,\; \text{clip}(\rho_i, 1-\varepsilon, 1+\varepsilon) \hat{A}_i\right)$$

**What this does:** if the advantage is positive (we want to increase the
probability of $y_i$), the clip prevents $\rho_i$ from growing above
$1 + \varepsilon$ — limiting how much we boost that response. If the
advantage is negative, it prevents $\rho_i$ from falling below
$1 - \varepsilon$. The min ensures we always take the more conservative
(smaller) update.

### The KL penalty

On top of clipping, GRPO adds a KL divergence penalty against the reference
model (the SFT model, as in DPO):

$$\mathcal{L}_{\text{GRPO}}(\theta) = -\frac{1}{G} \sum_{i=1}^G \mathcal{L}_{\text{clip},i}(\theta) + \beta \, D_{\text{KL}}\!\left(\pi_\theta \,\|\, \pi_{\text{ref}}\right)$$

The KL term is estimated per-token and averaged over the response:

$$D_{\text{KL}}\!\left(\pi_\theta \,\|\, \pi_{\text{ref}}\right) \approx \frac{1}{|y|} \sum_{t} \left[\log \frac{\pi_\theta(y_t \mid x, y_{<t})}{\pi_{\text{ref}}(y_t \mid x, y_{<t})}\right]$$

For efficiency, we use the unbiased single-sample estimator:

$$D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}}) \approx \frac{\pi_\theta(y_t)}{\pi_{\text{ref}}(y_t)} - \log \frac{\pi_\theta(y_t)}{\pi_{\text{ref}}(y_t)} - 1$$

This estimator is always non-negative (from the inequality $x - \log x \geq 1$)
and is unbiased.

In [ ]:
def kl_divergence_estimate(
    policy_log_probs: torch.Tensor,   # (B, T) log p_theta(token)
    ref_log_probs:    torch.Tensor,   # (B, T) log p_ref(token)
    mask:             torch.Tensor,   # (B, T) 1 for response tokens
) -> torch.Tensor:
    """
    Per-sequence KL divergence estimate using the unbiased estimator:
        KL(pi || ref) ≈ ratio - log(ratio) - 1
    where ratio = pi(y) / ref(y).

    Returns (B,) KL estimates, averaged over response tokens.
    """
    log_ratio = policy_log_probs - ref_log_probs   # (B, T)
    ratio     = log_ratio.exp()
    kl_per_token = ratio - log_ratio - 1            # (B, T), always >= 0

    # Average over response tokens only
    n_tokens = mask.sum(dim=1).clamp(min=1)         # (B,)
    return (kl_per_token * mask).sum(dim=1) / n_tokens   # (B,)

---

## 4. Implementation

### Step 1: Sample G responses per prompt

In [ ]:
import torch
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class GRPOConfig:
    G:               int   = 8       # responses per prompt
    clip_eps:        float = 0.2     # PPO clip epsilon
    beta:            float = 0.04    # KL penalty coefficient
    max_new_tokens:  int   = 128     # max response length
    temperature:     float = 0.9     # sampling temperature
    max_lr:          float = 1e-6    # very low — policy is already good
    min_lr:          float = 1e-7
    warmup_steps:    int   = 10
    max_steps:       int   = 200
    batch_prompts:   int   = 4       # prompts per step
    grad_clip:       float = 1.0
    eval_every:      int   = 50


@torch.no_grad()
def sample_responses(
    model,
    tokenizer,
    prompt_ids:     torch.Tensor,   # (1, T_prompt)
    G:              int,
    max_new_tokens: int,
    temperature:    float,
) -> torch.Tensor:
    """
    Sample G responses from the model for a single prompt.
    Returns response_ids: (G, T_response) — response tokens only.
    """
    model.eval()
    prompt_len = prompt_ids.size(1)

    # Repeat prompt G times for batched generation
    input_ids  = prompt_ids.repeat(G, 1)   # (G, T_prompt)

    output_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        eos_token_id=tokenizer.eos_id,
        do_sample=True,
    )   # (G, T_prompt + T_response)

    # Return only the response tokens
    return output_ids[:, prompt_len:]   # (G, T_response)

### Step 2: Score the responses

In [ ]:
@torch.no_grad()
def score_responses(
    reward_model,
    tokenizer,
    prompt_ids:    torch.Tensor,   # (1, T_prompt)
    response_ids:  torch.Tensor,   # (G, T_response)
    device:        torch.device,
) -> torch.Tensor:
    """
    Score G responses using the reward model.
    Returns rewards: (G,)
    """
    reward_model.eval()
    G          = response_ids.size(0)
    prompt_rep = prompt_ids.repeat(G, 1)   # (G, T_prompt)

    # Concatenate prompt + response for each of G samples
    full_ids = torch.cat([prompt_rep, response_ids], dim=1)   # (G, T_total)

    rewards = reward_model(full_ids)   # (G,)
    return rewards

### Step 3: Compute log-probabilities under policy and reference

In [ ]:
def get_response_log_probs(
    model,
    prompt_ids:   torch.Tensor,   # (G, T_prompt)
    response_ids: torch.Tensor,   # (G, T_response)
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute per-token log-probabilities for response tokens.

    Returns:
        log_probs: (G, T_response) — log p(token) for each response token
        mask:      (G, T_response) — 1 for real tokens, 0 for padding
    """
    G          = response_ids.size(0)
    full_ids   = torch.cat([prompt_ids, response_ids], dim=1)  # (G, T_total)
    T_prompt   = prompt_ids.size(1)
    T_response = response_ids.size(1)

    logits, _ = model(full_ids)   # (G, T_total, V)

    # Logits at position t predict token at position t+1
    # For response tokens at positions [T_prompt, T_prompt+T_response):
    # logits[:, T_prompt-1 : T_prompt+T_response-1, :] predict response tokens
    response_logits = logits[:, T_prompt-1 : T_prompt+T_response-1, :]  # (G, T_r, V)

    log_probs_all = F.log_softmax(response_logits, dim=-1)  # (G, T_r, V)

    # Gather log p for the actual response tokens
    token_log_probs = log_probs_all.gather(
        dim=2,
        index=response_ids.unsqueeze(2)
    ).squeeze(2)   # (G, T_response)

    # Mask: 0 wherever response_ids is the pad token
    # For simplicity assume pad_id = 0; adjust for your tokenizer
    mask = (response_ids != tokenizer.pad_id).float()   # (G, T_response)

    return token_log_probs, mask

### Step 4: The GRPO loss

In [ ]:
def grpo_loss(
    policy_log_probs:    torch.Tensor,   # (G, T_response)
    old_log_probs:       torch.Tensor,   # (G, T_response) — from sampling step
    ref_log_probs:       torch.Tensor,   # (G, T_response) — frozen ref model
    advantages:          torch.Tensor,   # (G,)
    mask:                torch.Tensor,   # (G, T_response)
    clip_eps:            float = 0.2,
    beta:                float = 0.04,
) -> tuple[torch.Tensor, dict]:
    """
    GRPO loss for G responses to a single prompt.

    Returns (loss, metrics_dict).
    """
    G, T = policy_log_probs.shape

    # ---- Importance ratios ----
    # log rho = log pi_theta(y) - log pi_theta_old(y)
    # Sum over tokens gives log of the sequence-level ratio
    log_ratio     = (policy_log_probs - old_log_probs) * mask  # (G, T)
    seq_log_ratio = log_ratio.sum(dim=1)                        # (G,)
    ratio         = seq_log_ratio.exp()                         # (G,)

    # ---- Clipped surrogate ----
    adv = advantages   # (G,)

    surr_unclipped = ratio * adv
    surr_clipped   = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
    surrogate      = torch.min(surr_unclipped, surr_clipped)   # (G,)

    policy_loss = -surrogate.mean()

    # ---- KL penalty ----
    kl = kl_divergence_estimate(policy_log_probs, ref_log_probs, mask)  # (G,)
    kl_loss = beta * kl.mean()

    # ---- Total loss ----
    loss = policy_loss + kl_loss

    # ---- Metrics ----
    n_tokens   = mask.sum().item()
    clipped_frac = ((ratio - 1).abs() > clip_eps).float().mean().item()

    metrics = {
        'loss':          loss.item(),
        'policy_loss':   policy_loss.item(),
        'kl_loss':       kl_loss.item(),
        'mean_kl':       kl.mean().item(),
        'mean_reward':   0.0,   # filled in by the caller
        'mean_ratio':    ratio.mean().item(),
        'clipped_frac':  clipped_frac,
        'mean_advantage': adv.mean().item(),
        'reward_std':    0.0,   # filled in by the caller
    }
    return loss, metrics

---

## 5. The GRPO Training Loop

The GRPO loop has a fundamentally different structure from SFT:

```
SFT loop:
  for each batch:
    forward pass → loss → backward → step

GRPO loop:
  for each batch of prompts:
    1. sample G responses per prompt  (no grad)
    2. score responses with RM        (no grad)
    3. compute advantages             (no grad)
    4. compute old log-probs          (no grad, freeze snapshot)
    5. compute ref log-probs          (no grad, frozen ref model)
    6. forward pass with grad         (policy_log_probs)
    7. compute GRPO loss              (uses ratio = policy/old)
    8. backward → step
```

Steps 1–5 are data preparation — they generate the training signal for
this step. Step 6–8 are the actual gradient update.

In [ ]:
import copy
import time
from pathlib import Path
from torch.utils.data import DataLoader

def grpo_train(
    policy_path:    str,
    ref_path:       str,
    reward_path:    str,
    prompt_dataset,          # dataset that yields prompt strings
    output_dir:     str,
    cfg:            GRPOConfig = None,
):
    if cfg is None:
        cfg = GRPOConfig()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.bfloat16 if device.type == 'cuda' else torch.float32
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    from tutorial_02 import GPT, NanoGPTConfig
    from tutorial_03 import Tokenizer

    config = NanoGPTConfig()
    tok    = Tokenizer.load('nano_tokenizer.json')

    # ---- Policy model (trained) ----
    policy = GPT(config).to(device)
    ckpt   = torch.load(policy_path, map_location=device)
    policy.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)

    # ---- Reference model (frozen SFT model) ----
    ref_model = GPT(config).to(device)
    ref_model.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)

    # ---- Reward model ----
    rm_ckpt      = torch.load(reward_path, map_location=device)
    reward_model = RewardModel(config).to(device)
    reward_model.load_state_dict(rm_ckpt['model'])
    reward_model.eval()
    for p in reward_model.parameters():
        p.requires_grad_(False)

    # ---- Optimizer — very low LR, policy is already good ----
    optimizer = torch.optim.AdamW(
        policy.parameters(), lr=cfg.max_lr, weight_decay=0.01
    )
    scheduler = make_cosine_schedule(
        optimizer, cfg.max_lr, cfg.min_lr, cfg.warmup_steps, cfg.max_steps
    )

    # ---- Training loop ----
    prompt_loader = DataLoader(
        prompt_dataset, batch_size=cfg.batch_prompts, shuffle=True
    )
    prompt_iter = iter(prompt_loader)
    history     = []

    for step in range(cfg.max_steps):
        try:
            prompts = next(prompt_iter)
        except StopIteration:
            prompt_iter = iter(prompt_loader)
            prompts     = next(prompt_iter)

        step_metrics = []

        optimizer.zero_grad()
        total_loss = torch.tensor(0.0, device=device)

        for prompt_text in prompts:
            # ---- Encode prompt ----
            prompt_enc = (f"{SPECIAL_TOKENS['user']}\n{prompt_text}\n"
                          f"{SPECIAL_TOKENS['end']}\n"
                          f"{SPECIAL_TOKENS['assistant']}\n")
            prompt_ids = torch.tensor(
                [tok.encode(prompt_enc)], dtype=torch.long, device=device
            )   # (1, T_prompt)

            # ---- Step 1: Sample G responses ----
            with torch.no_grad():
                response_ids = sample_responses(
                    policy, tok, prompt_ids,
                    G=cfg.G,
                    max_new_tokens=cfg.max_new_tokens,
                    temperature=cfg.temperature,
                )   # (G, T_response)

            # ---- Step 2: Score with reward model ----
            with torch.no_grad():
                rewards = score_responses(
                    reward_model, tok, prompt_ids, response_ids, device
                )   # (G,)

            # ---- Step 3: Compute advantages ----
            advantages = compute_group_advantages(rewards)   # (G,)

            # ---- Step 4: Old log-probs (policy at sampling time) ----
            prompt_rep = prompt_ids.repeat(cfg.G, 1)   # (G, T_prompt)
            with torch.no_grad():
                old_log_probs, mask = get_response_log_probs(
                    policy, prompt_rep, response_ids
                )   # (G, T_response)

            # ---- Step 5: Reference log-probs ----
            with torch.no_grad():
                ref_log_probs, _ = get_response_log_probs(
                    ref_model, prompt_rep, response_ids
                )   # (G, T_response)

            # ---- Step 6: Policy log-probs WITH gradient ----
            policy.train()
            with torch.autocast(device_type=device.type, dtype=dtype):
                policy_log_probs, _ = get_response_log_probs(
                    policy, prompt_rep, response_ids
                )   # (G, T_response)

            # ---- Step 7: GRPO loss ----
            loss, metrics = grpo_loss(
                policy_log_probs, old_log_probs, ref_log_probs,
                advantages, mask,
                clip_eps=cfg.clip_eps, beta=cfg.beta,
            )
            metrics['mean_reward'] = rewards.mean().item()
            metrics['reward_std']  = rewards.std().item()

            # Normalize loss by number of prompts in batch
            (loss / len(prompts)).backward()
            total_loss     += loss.detach()
            step_metrics.append(metrics)

        # ---- Step 8: Gradient step ----
        torch.nn.utils.clip_grad_norm_(policy.parameters(), cfg.grad_clip)
        optimizer.step()
        scheduler.step()

        # ---- Aggregate metrics ----
        avg = {k: np.mean([m[k] for m in step_metrics]) for k in step_metrics[0]}
        history.append(avg)

        if step % 10 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(
                f"step {step:4d}  "
                f"reward={avg['mean_reward']:+.3f}  "
                f"kl={avg['mean_kl']:.4f}  "
                f"adv={avg['mean_advantage']:+.3f}  "
                f"clip%={avg['clipped_frac']:.1%}  "
                f"lr={lr:.1e}"
            )

        # ---- Checkpoint ----
        if step % cfg.eval_every == 0 and step > 0:
            torch.save({
                'step':    step,
                'model':   policy.state_dict(),
                'metrics': avg,
            }, f'{output_dir}/grpo_step{step:04d}.pt')

    return policy, history

---

## 6. Monitoring GRPO Training

GRPO has more failure modes than SFT. Monitor all of these:

### Mean reward

Should increase over training. If it plateaus immediately: the reward
model may not have enough signal to distinguish the G sampled responses
(reward std is near zero, all advantages are near zero, no learning signal).

### KL divergence

Should stay small. Healthy range for our setup: $[0.01, 0.5]$.

- [**KL < 0.01**: policy is barely moving.]{.underline} β may be too high, or LR too low.
- **KL > 1.0**: policy is diverging from the reference. Reduce β or LR.
- [[[**KL growing monotonically**: reward hacking in progress]{.mark} — stop training.]{.mark}

### Clipped fraction

Fraction of importance ratios that hit the clip boundary. Healthy: 10–30%.

- **Clipped fraction > 50%**: updates are being severely truncated —
  LR is too high or β is too low.
- **Clipped fraction < 5%**: updates are all within clip range — LR may
  be too low, or the policy has converged.

### Reward standard deviation within group

If `reward_std ≈ 0` for most prompts, all G responses score similarly —
the advantage signal is near zero and the policy cannot learn. Solutions:
increase sampling temperature, use a better-calibrated reward model, or
reduce the group size $G$ (fewer samples means more variance in the mean).

### Response entropy

Measures diversity of the generated responses. Compute the entropy of the
token distribution at each position:

In [ ]:
@torch.no_grad()
def response_entropy(
    model,
    prompt_ids: torch.Tensor,   # (1, T_prompt)
    n_samples:  int = 32,
    max_tokens: int = 50,
    temperature: float = 1.0,
) -> float:
    """
    Estimate mean per-token entropy of the policy over response positions.
    Low entropy = model is becoming deterministic (collapse risk).
    """
    model.eval()
    prompt_rep = prompt_ids.repeat(n_samples, 1)
    outputs    = model.generate(
        prompt_rep, max_new_tokens=max_tokens, temperature=temperature,
        eos_token_id=None, do_sample=True, output_scores=True,
    )
    # output_scores: list of (n_samples, vocab_size) per step
    if not hasattr(outputs, 'scores') or outputs.scores is None:
        return float('nan')

    entropies = []
    for score in outputs.scores:
        probs = F.softmax(score, dim=-1)                   # (n_samples, V)
        ent   = -(probs * (probs + 1e-10).log()).sum(-1)   # (n_samples,)
        entropies.append(ent.mean().item())

    return float(np.mean(entropies))

In [ ]:
def plot_grpo_training(history: list[dict]):
    import matplotlib.pyplot as plt

    steps = list(range(len(history)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('GRPO Training Diagnostics', fontsize=14, fontweight='bold')

    panels = [
        ('mean_reward',   'Mean Reward',         '#4CAF50'),
        ('mean_kl',       'Mean KL Divergence',  '#F44336'),
        ('clipped_frac',  'Clipped Fraction',    '#FF9800'),
        ('reward_std',    'Reward Std (group)',   '#2196F3'),
        ('mean_advantage','Mean Advantage',       '#9C27B0'),
        ('policy_loss',   'Policy Loss',          '#607D8B'),
    ]
    for ax, (key, title, color) in zip(axes.flat, panels):
        ax.plot(steps, [m[key] for m in history], color=color, lw=1.5)
        ax.set_title(title); ax.set_xlabel('Step')
        if key == 'mean_kl':
            ax.axhline(0.5, color='gray', linestyle='--', lw=0.8, label='KL=0.5')
            ax.legend()
        if key == 'clipped_frac':
            ax.axhline(0.3, color='gray', linestyle='--', lw=0.8)
            ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('grpo_training.png', dpi=150)
    plt.show()

---

## 7. Reward Hacking

[Reward hacking is the central failure mode of GRPO.]{.underline} The policy finds
behaviors that score highly on the reward model but do not correspond
to actually good responses. Because the reward model is imperfect, there
are always such behaviors.

Common reward hacking patterns:

**Length exploitation:** if the reward model has length bias (Tutorial 13),
the policy learns to generate very long responses. Detect with the
`check_length_bias` probe run periodically during GRPO.

**Format exploitation:** the reward model may give higher scores to responses
with specific formatting (bullet points, numbered lists, bold text). The
policy learns to always use these formats regardless of whether they help.

**Repetition:** a degenerate policy that repeats the same high-scoring phrase
many times. The reward model often rewards this because the scoring happens
at the last token position and repeated phrases can dominate the hidden state.

**KL explosion:** the policy diverges so far from the reference that it
generates fluent-looking but semantically empty text that happens to fool
the reward model.

In [ ]:
def detect_reward_hacking(
    history:          list[dict],
    policy,
    reward_model,
    tokenizer,
    calibration_prompts: list[str],
    device:           torch.device,
    step:             int,
) -> list[str]:
    """
    Run a battery of reward hacking checks.
    Returns a list of warning strings (empty if healthy).
    """
    warnings = []

    # Check 1: KL divergence explosion
    if len(history) >= 10:
        recent_kl = np.mean([m['mean_kl'] for m in history[-10:]])
        if recent_kl > 1.0:
            warnings.append(
                f"⚠ KL={recent_kl:.3f} > 1.0 — policy diverging from reference"
            )

    # Check 2: Reward still growing after KL has plateaued
    # (suggests reward hacking rather than genuine improvement)
    if len(history) >= 20:
        early_kl   = np.mean([m['mean_kl']     for m in history[:10]])
        recent_kl  = np.mean([m['mean_kl']     for m in history[-10:]])
        early_r    = np.mean([m['mean_reward']  for m in history[:10]])
        recent_r   = np.mean([m['mean_reward']  for m in history[-10:]])
        kl_growth  = recent_kl  / (early_kl + 1e-8)
        r_growth   = recent_r   / (early_r  + 1e-8)
        if r_growth > 2.0 and kl_growth > 3.0:
            warnings.append(
                f"⚠ Reward growing fast ({r_growth:.1f}×) with KL explosion "
                f"({kl_growth:.1f}×) — likely reward hacking"
            )

    # Check 3: Response diversity (repetition detection)
    policy.eval()
    with torch.no_grad():
        for prompt in calibration_prompts[:3]:
            enc = tok.encode(
                f"{SPECIAL_TOKENS['user']}\n{prompt}\n"
                f"{SPECIAL_TOKENS['end']}\n{SPECIAL_TOKENS['assistant']}\n"
            )
            ids = torch.tensor([enc], dtype=torch.long, device=device)
            out = policy.generate(ids, max_new_tokens=80, temperature=0.7,
                                  eos_token_id=tok.eos_id)
            response = tok.decode(out[0, len(enc):].tolist())

            # Check for repetition: any 5-gram appearing more than 3 times
            words  = response.split()
            ngrams = [' '.join(words[i:i+5]) for i in range(len(words)-4)]
            if ngrams:
                from collections import Counter
                most_common_count = Counter(ngrams).most_common(1)[0][1]
                if most_common_count > 3:
                    warnings.append(
                        f"⚠ Repetition detected in response to '{prompt[:30]}...'"
                    )

    return warnings

### Early stopping for GRPO

Because reward hacking can be subtle and hard to reverse, add automatic
early stopping based on KL divergence:

In [ ]:
class GRPOEarlyStopper:
    """
    Stops GRPO training if KL divergence exceeds a threshold
    or if reward has not improved in patience steps.
    """

    def __init__(
        self,
        kl_threshold:      float = 1.0,
        reward_patience:   int   = 30,
        min_reward_delta:  float = 0.01,
    ):
        self.kl_threshold    = kl_threshold
        self.reward_patience = reward_patience
        self.min_reward_delta = min_reward_delta
        self.best_reward     = -float('inf')
        self.steps_no_improve = 0

    def check(self, metrics: dict) -> tuple[bool, str]:
        """
        Returns (should_stop, reason).
        """
        kl     = metrics.get('mean_kl', 0)
        reward = metrics.get('mean_reward', 0)

        if kl > self.kl_threshold:
            return True, f"KL={kl:.3f} exceeded threshold {self.kl_threshold}"

        if reward > self.best_reward + self.min_reward_delta:
            self.best_reward      = reward
            self.steps_no_improve = 0
        else:
            self.steps_no_improve += 1
            if self.steps_no_improve >= self.reward_patience:
                return True, (f"No reward improvement for "
                              f"{self.reward_patience} steps "
                              f"(best={self.best_reward:.3f})")

        return False, ""

---

## 8. GRPO vs PPO: What We Skipped and Why

PPO maintains a **value network** — a separate model that estimates
$V(x) = \mathbb{E}_{y \sim \pi}[r(x,y)]$ for each prompt. The advantage
is then $A(x,y) = r(y) - V(x)$ — a per-response estimate that the value
network provides without needing to sample multiple responses.

GRPO replaces the value network with the group mean. This is:

- **Simpler**: no second model to train, no bootstrapping, no GAE
- **Cheaper**: no value network forward/backward pass
- **Slightly noisier**: with $G=8$ samples, the group mean is a noisier
  estimate of $V(x)$ than a trained value network

For language model fine-tuning at the scale covered in this series, GRPO's
simplicity wins. PPO's value network advantage matters most when:
- $G$ is very small (< 4) — group mean has high variance
- The reward signal is very sparse (most responses score identically)
- You are running thousands of GRPO steps and need variance reduction

For the nano model with a clean reward model and $G=8$, GRPO is the
right choice.

---

## Summary

| Concept | Key detail |
|---|---|
| REINFORCE | $\nabla J = \mathbb{E}[r(y) \cdot \nabla \log \pi(y\|x)]$. Log-derivative trick. |
| Baseline | Any $b(x)$ not depending on $y$ can be subtracted without bias. Reduces variance. |
| Group advantage | $\hat{A}_i = (r_i - \text{mean}(r)) / \text{std}(r)$. No value network needed. |
| Importance ratio | $\rho = \pi_\theta(y) / \pi_{\text{old}}(y)$. Sequence-level: sum log-probs then exp. |
| Clipped surrogate | $\min(\rho \hat{A},\, \text{clip}(\rho, 1\pm\varepsilon)\hat{A})$. Prevents large policy jumps. |
| KL estimator | $\rho - \log\rho - 1$. Always non-negative. Unbiased. No second forward pass. |
| Loop structure | Sample → score → advantages → old log-probs → ref log-probs → grad update. |
| $\beta$ | KL penalty. Typical: 0.04. Controls how far policy drifts from reference. |
| $\varepsilon$ (clip) | Clip threshold. Typical: 0.2. Smaller = more conservative updates. |
| $G$ (group size) | Responses per prompt. More = lower variance advantages. Typical: 4–16. |
| Reward hacking | KL explosion + reward growth = policy fooling the RM, not improving. |
| Early stopping | Stop when KL > threshold or reward plateaus. Save the checkpoint before hacking begins. |
| GRPO vs PPO | GRPO replaces value network with group mean. Simpler, slightly noisier. |

---

## Exercises

**1.** Derive the REINFORCE estimator. Starting from
$J(\theta) = \sum_y r(y) \pi_\theta(y \mid x)$, apply the log-derivative
trick step by step and arrive at
$\nabla_\theta J = \mathbb{E}_{y \sim \pi_\theta}[r(y) \nabla_\theta \log \pi_\theta(y \mid x)]$.
Then show that subtracting any baseline $b(x)$ leaves the gradient
unbiased by proving $\mathbb{E}[b(x) \nabla_\theta \log \pi_\theta] = 0$.

**2.** Implement `verify_importance_ratio`: given a policy and a set of
sampled responses, compute the sequence-level importance ratio
$\rho = \exp(\sum_t \log \pi_\theta(y_t) - \log \pi_{\text{old}}(y_t))$
two ways: (a) sum log-probs then exp, (b) multiply per-token ratios
$\prod_t (\pi_\theta(y_t) / \pi_{\text{old}}(y_t))$. Verify they
are numerically equal. Then show why method (b) is numerically unstable
for sequences of length > 20 (underflow/overflow in float32).

**3.** Implement the **group size ablation**: run GRPO for 100 steps with
$G \in \{2, 4, 8, 16\}$. For each $G$, record the mean reward and the
variance of the advantage estimates per step. Plot reward vs step and
advantage variance vs $G$. Confirm that larger $G$ reduces advantage
variance and produces smoother reward curves, at the cost of more
sampling compute.

**4.** The KL estimator $\rho - \log\rho - 1$ is unbiased and non-negative.
Verify both properties numerically: (a) sample 10000 values of $\log\rho$
from $\mathcal{N}(0, 0.5)$ (typical in early training), compute the
estimator, and verify its mean is close to the true KL of that distribution;
(b) verify that all 10000 values of the estimator are $\geq 0$.

**5.** Implement a reward hacking detector that monitors response length
every 10 steps during GRPO. Plot mean response length vs step alongside
mean reward. If the reward model has length bias, you should see both
curves increasing together — the policy is gaming the RM rather than
improving. Confirm this by running GRPO with a deliberately length-biased
reward model (one trained without the length penalty from Tutorial 13
Exercise 3).

**6.** Implement **GRPO with process reward**: instead of a single outcome
reward $r(x, y)$ for the whole response, assign a per-step reward
$r_t$ at each token position (e.g., using a trained process reward model
or a simple heuristic like grammar score per sentence). The advantage
becomes $\hat{A}_{i,t} = r_{i,t} - \text{mean}_j(r_{j,t})$ at each
position. Modify `grpo_loss` to accept per-token rewards and compute
per-token advantages. This is the setup used in process-supervised GRPO
for mathematical reasoning.